# Temporal Reasoning Micro Tests

> Quick tests for the LLM Temporal Reasoning capabilities

In [ ]:
#| default_exp core.temporal


In [ ]:
#| export
from __future__ import annotations
from fastcore.test import test_ne, test_eq
from datetime import datetime, timezone, timedelta
import json

In [ ]:
#| export
from typing import Union, List, Dict, Tuple, Optional
from pydantic import BaseModel, ConfigDict
from rdflib import Graph, Namespace, URIRef, Literal as RDFLiteral, BNode
from rdflib.namespace import RDF, RDFS, OWL, XSD, TIME
from pyshacl import validate
from enum import Enum
from cogitarelink.core.graph import GraphManager
from cogitarelink.core.debug import get_logger
from cogitarelink.reason.prov import wrap_patch_with_prov

In [ ]:
#| export
log = get_logger("temporal")

In [ ]:
#| export
# Define our namespaces
class Namespaces:
    EVENT = Namespace("https://example.org/event-ontology#")
    OLIVE = Namespace("https://example.org/olive-temporal#")
    TEMP = Namespace("https://example.org/temporal-relations#")

In [ ]:
#| export
def ensure_timezone(dt):
    "Ensure a datetime has timezone information (UTC if none)"
    if dt.tzinfo is None: return dt.replace(tzinfo=timezone.utc)
    return dt

class TimeInstant(BaseModel):
    "Represents a specific moment in time"
    dateTime: datetime
    granularity: Optional[str] = "second"
    description: Optional[str] = None
    
    model_config = ConfigDict(arbitrary_types_allowed=True)
    
    def __init__(self, **data):
        super().__init__(**data)
        self.dateTime = ensure_timezone(self.dateTime)
        
    def to_graph(self, g=None, node=None):
        "Add this instant to an RDF graph"
        if g is None: g = Graph()
        if node is None: node = BNode()
        
        g.add((node, RDF.type, TIME.Instant))
        g.add((node, TIME.inXSDDateTime, RDFLiteral(self.dateTime.isoformat(), datatype=XSD.dateTime)))
        if self.description: g.add((node, RDFS.comment, RDFLiteral(self.description)))
        
        return g, node

In [ ]:
#| export
class TimeInterval(BaseModel):
    "Represents a period of time with a start and end"
    startTime: Union[TimeInstant, Dict]
    endTime: Union[TimeInstant, Dict] 
    description: Optional[str] = None
    
    model_config = ConfigDict(arbitrary_types_allowed=True)
    
    def __init__(self, **data):
        super().__init__(**data)
        if isinstance(self.startTime, dict): self.startTime = TimeInstant(**self.startTime)
        if isinstance(self.endTime, dict): self.endTime = TimeInstant(**self.endTime)
    
    def duration(self) -> timedelta: 
        return self.endTime.dateTime - self.startTime.dateTime
    
    def to_graph(self, g=None, node=None):
        "Add this interval to an RDF graph with proper OWL-Time structure"
        if g is None: g = Graph()
        if node is None: node = BNode()
        
        # Add interval type
        g.add((node, RDF.type, TIME.Interval))
        
        # Add beginning instant
        start_node = BNode()
        g, start_node = self.startTime.to_graph(g, start_node)
        g.add((node, TIME.hasBeginning, start_node))
        
        # Add end instant
        end_node = BNode()
        g, end_node = self.endTime.to_graph(g, end_node)
        g.add((node, TIME.hasEnd, end_node))
        
        # Add description if available
        if self.description: g.add((node, RDFS.comment, RDFLiteral(self.description)))
        
        return g, node

In [ ]:
#| export
class InstantReification(BaseModel):
    "Reification of a relationship at a specific moment in time"
    subject: str
    predicate: str
    object: str
    timeInstant: TimeInstant
    properties: Optional[Dict[str, Union[str, int, float, bool]]] = {}
    
    model_config = ConfigDict(arbitrary_types_allowed=True)
    
    def __init__(self, **data):
        if isinstance(data.get('timeInstant'), dict): data['timeInstant'] = TimeInstant(**data['timeInstant'])
        super().__init__(**data)
        
    def to_graph(self, g=None, node=None):
        "Add this reification to an RDF graph"
        if g is None: g = Graph()
        if node is None: node = BNode()
        
        # Add reification type
        g.add((node, RDF.type, Namespaces.OLIVE.InstantReification))
        
        # Add subject, predicate, object
        g.add((node, Namespaces.OLIVE.subject, URIRef(self.subject)))
        g.add((node, Namespaces.OLIVE.predicate, URIRef(self.predicate))) 
        g.add((node, Namespaces.OLIVE.object, URIRef(self.object)))
        
        # Add time instant
        instant_node = BNode()
        g, instant_node = self.timeInstant.to_graph(g, instant_node)
        g.add((node, Namespaces.OLIVE.atTimeInstant, instant_node))
        
        # Add properties
        for k, v in self.properties.items():
            g.add((node, URIRef(f"{Namespaces.OLIVE}{k}"), RDFLiteral(v)))
            
        return g, node

In [ ]:
#| export
class IntervalReification(BaseModel):
    "Reification of a relationship over a time interval"
    subject: str
    predicate: str
    object: str
    timeInterval: TimeInterval
    properties: Optional[Dict[str, Union[str, int, float, bool]]] = {}
    
    model_config = ConfigDict(arbitrary_types_allowed=True)
    
    def __init__(self, **data):
        if isinstance(data.get('timeInterval'), dict): data['timeInterval'] = TimeInterval(**data['timeInterval'])
        super().__init__(**data)
        
    def to_graph(self, g=None, node=None):
        "Add this reification to an RDF graph"
        if g is None: g = Graph()
        if node is None: node = BNode()
        
        # Add reification type
        g.add((node, RDF.type, Namespaces.OLIVE.IntervalReification))
        
        # Add subject, predicate, object
        g.add((node, Namespaces.OLIVE.subject, URIRef(self.subject)))
        g.add((node, Namespaces.OLIVE.predicate, URIRef(self.predicate)))
        g.add((node, Namespaces.OLIVE.object, URIRef(self.object)))
        
        # Add time interval
        interval_node = BNode()
        g, interval_node = self.timeInterval.to_graph(g, interval_node)
        g.add((node, Namespaces.OLIVE.hasTimeInterval, interval_node))
        
        # Add properties
        for k, v in self.properties.items():
            g.add((node, URIRef(f"{Namespaces.OLIVE}{k}"), RDFLiteral(v)))
            
        return g, node

In [ ]:
#| export
class LifespanReification(BaseModel):
    "Reification of a relationship over its entire lifespan"
    subject: str
    predicate: str
    object: str
    properties: Optional[Dict[str, Union[str, int, float, bool]]] = {}
    
    model_config = ConfigDict(arbitrary_types_allowed=True)
    
    def to_graph(self, g=None, node=None):
        "Add this reification to an RDF graph"
        if g is None: g = Graph()
        if node is None: node = BNode()
        
        # Add reification type
        g.add((node, RDF.type, Namespaces.OLIVE.LifespanReification))
        
        # Add subject, predicate, object
        g.add((node, Namespaces.OLIVE.subject, URIRef(self.subject)))
        g.add((node, Namespaces.OLIVE.predicate, URIRef(self.predicate)))
        g.add((node, Namespaces.OLIVE.object, URIRef(self.object)))
        
        # Add properties
        for k, v in self.properties.items():
            g.add((node, URIRef(f"{Namespaces.OLIVE}{k}"), RDFLiteral(v)))
            
        return g, node

In [ ]:
#| export
class Event(BaseModel):
    "Core event model that supports temporal reification"
    id: str
    name: str
    description: Optional[str] = None
    startTime: TimeInstant
    endTime: Optional[TimeInstant] = None
    location: Optional[str] = None
    
    # Participants with their basic roles
    participants: List[Dict[str, str]] = []
    
    # Temporal reifications
    instantReifications: List[InstantReification] = []
    intervalReifications: List[IntervalReification] = []
    lifespanReifications: List[LifespanReification] = []
    
    model_config = ConfigDict(arbitrary_types_allowed=True)
    
    def add_participant(self, participant_id: str, name: Optional[str] = None, role: Optional[str] = None) -> None:
        "Add a participant to the event"
        self.participants.append({"id": participant_id, "name": name, "role": role})
    
    def add_instant_reification(self, subject: str, predicate: str, object: str, 
                               time_instant: TimeInstant, properties: Optional[Dict] = None) -> None:
        "Add an instant reification to the event"
        self.instantReifications.append(InstantReification(
            subject=subject, predicate=predicate, object=object, 
            timeInstant=time_instant, properties=properties or {}
        ))
    
    def add_interval_reification(self, subject: str, predicate: str, object: str, 
                            time_interval: TimeInterval, properties: Optional[Dict] = None) -> None:
        "Add an interval reification to the event"
        self.intervalReifications.append(IntervalReification(
            subject=subject, predicate=predicate, object=object, 
            timeInterval=time_interval, properties=properties or {}
        ))
    
    def add_lifespan_reification(self, subject: str, predicate: str, object: str, 
                              properties: Optional[Dict] = None) -> None:
        "Add a lifespan reification to the event"
        self.lifespanReifications.append(LifespanReification(
            subject=subject, predicate=predicate, object=object, 
            properties=properties or {}
        ))
        
    def to_graph(self):
        "Convert Event to RDF graph with temporal reifications and proper intervals"
        g = Graph()
        
        # Register namespaces
        g.bind("event", Namespaces.EVENT)
        g.bind("olive", Namespaces.OLIVE)
        g.bind("time", TIME)
        g.bind("temp", Namespaces.TEMP)
        
        # Add event triples
        event_uri = URIRef(self.id)
        g.add((event_uri, RDF.type, Namespaces.EVENT.Event))
        g.add((event_uri, RDFS.label, RDFLiteral(self.name)))
        if self.description: g.add((event_uri, RDFS.comment, RDFLiteral(self.description)))
        
        # Add start time
        start_node = BNode()
        g, start_node = self.startTime.to_graph(g, start_node)
        g.add((event_uri, Namespaces.EVENT.hasStartTime, start_node))
        
        # Add end time if present
        end_node = None
        if self.endTime:
            end_node = BNode()
            g, end_node = self.endTime.to_graph(g, end_node)
            g.add((event_uri, Namespaces.EVENT.hasEndTime, end_node))
        
        # Create proper time:Interval for the event
        interval_node = BNode()
        g.add((interval_node, RDF.type, TIME.Interval))
        g.add((interval_node, TIME.hasBeginning, start_node))
        if end_node: g.add((interval_node, TIME.hasEnd, end_node))
        g.add((event_uri, Namespaces.EVENT.hasTimeInterval, interval_node))
        
        # Add location if present
        if self.location: g.add((event_uri, Namespaces.EVENT.hasLocation, URIRef(self.location)))
        
        # Add participants
        for p in self.participants:
            participant_uri = URIRef(p["id"])
            g.add((event_uri, Namespaces.EVENT.hasParticipant, participant_uri))
            if p.get("name"): g.add((participant_uri, RDFS.label, RDFLiteral(p["name"])))
            if p.get("role"): g.add((participant_uri, Namespaces.EVENT.hasRole, RDFLiteral(p["role"])))
        
        # Add instant reifications
        for r in self.instantReifications:
            reif_node = BNode()
            g, reif_node = r.to_graph(g, reif_node)
            g.add((event_uri, Namespaces.OLIVE.hasInstantReification, reif_node))
        
        # Add interval reifications
        for r in self.intervalReifications:
            reif_node = BNode()
            g, reif_node = r.to_graph(g, reif_node)
            g.add((event_uri, Namespaces.OLIVE.hasIntervalReification, reif_node))
        
        # Add lifespan reifications
        for r in self.lifespanReifications:
            reif_node = BNode()
            g, reif_node = r.to_graph(g, reif_node)
            g.add((event_uri, Namespaces.OLIVE.hasLifespanReification, reif_node))
        
        return g

In [ ]:
#| export
def create_temporal_rules():
    "Create SPARQL rules for temporal reasoning"
    inference_rules = """
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix time: <http://www.w3.org/2006/time#> .
@prefix temp: <https://example.org/temporal-relations#> .

# Shape that targets all time intervals
temp:IntervalShape
    a sh:NodeShape ;
    sh:targetClass time:Interval ;
    sh:rule temp:beforeRule, temp:afterRule, temp:containsRule, temp:duringRule .

temp:beforeRule a sh:SPARQLRule ;
    sh:construct '''
        PREFIX time: <http://www.w3.org/2006/time#>
        PREFIX temp: <https://example.org/temporal-relations#>
        
        CONSTRUCT {
            ?interval1 temp:before ?interval2 .
        }
        WHERE {
            ?interval1 time:hasEnd/time:inXSDDateTime ?end1 .
            ?interval2 time:hasBeginning/time:inXSDDateTime ?start2 .
            FILTER (?end1 < ?start2)
            FILTER (?interval1 != ?interval2)
        }
    ''' .

temp:afterRule a sh:SPARQLRule ;
    sh:construct '''
        PREFIX time: <http://www.w3.org/2006/time#>
        PREFIX temp: <https://example.org/temporal-relations#>
        
        CONSTRUCT {
            ?interval1 temp:after ?interval2 .
        }
        WHERE {
            ?interval1 time:hasBeginning/time:inXSDDateTime ?start1 .
            ?interval2 time:hasEnd/time:inXSDDateTime ?end2 .
            FILTER (?start1 > ?end2)
            FILTER (?interval1 != ?interval2)
        }
    ''' .

temp:containsRule a sh:SPARQLRule ;
    sh:construct '''
        PREFIX time: <http://www.w3.org/2006/time#>
        PREFIX temp: <https://example.org/temporal-relations#>
        
        CONSTRUCT {
            ?interval1 temp:contains ?interval2 .
        }
        WHERE {
            ?interval1 time:hasBeginning/time:inXSDDateTime ?start1 .
            ?interval1 time:hasEnd/time:inXSDDateTime ?end1 .
            ?interval2 time:hasBeginning/time:inXSDDateTime ?start2 .
            ?interval2 time:hasEnd/time:inXSDDateTime ?end2 .
            FILTER (?start1 <= ?start2 && ?end1 >= ?end2)
            FILTER (?interval1 != ?interval2)
        }
    ''' .

temp:duringRule a sh:SPARQLRule ;
    sh:construct '''
        PREFIX time: <http://www.w3.org/2006/time#>
        PREFIX temp: <https://example.org/temporal-relations#>
        
        CONSTRUCT {
            ?interval1 temp:during ?interval2 .
        }
        WHERE {
            ?interval1 time:hasBeginning/time:inXSDDateTime ?start1 .
            ?interval1 time:hasEnd/time:inXSDDateTime ?end1 .
            ?interval2 time:hasBeginning/time:inXSDDateTime ?start2 .
            ?interval2 time:hasEnd/time:inXSDDateTime ?end2 .
            FILTER (?start1 >= ?start2 && ?end1 <= ?end2)
            FILTER (?interval1 != ?interval2)
        }
    ''' .
"""
    return inference_rules

def infer_temporal_relations(event_graph):
    "Apply temporal reasoning rules to infer relationships between intervals"
    # Create rules graph with properly attached shapes
    rules_graph = Graph()
    rules_graph.parse(data=create_temporal_rules(), format='turtle')
    
    # Apply SHACL rules
    result = validate(
        event_graph,
        shacl_graph=rules_graph,
        advanced=True,
        inference='rdfs',
        iterate_rules=True,
        inplace=True
    )
    
    # Return the graph with inferred relationships
    return event_graph

In [ ]:
#| export
def create_jsonld_context():
    "Create a JSON-LD context for our event model"
    return {
        "@context": {
            "@version": 1.1,
            "@vocab": "https://example.org/event-ontology#",
            "time": "http://www.w3.org/2006/time#",
            "event": "https://example.org/event-ontology#",
            "olive": "https://example.org/olive-temporal#",
            "temp": "https://example.org/temporal-relations#",
            
            "id": "@id",
            "type": "@type",
            "Event": "event:Event",
            "name": {"@id": "rdfs:label", "@type": "xsd:string"},
            "description": {"@id": "rdfs:comment", "@type": "xsd:string"},
            "location": {"@id": "event:hasLocation", "@type": "@id"},
            
            "startTime": {"@id": "event:hasStartTime", "@type": "TimeInstant"},
            "endTime": {"@id": "event:hasEndTime", "@type": "TimeInstant"},
            "timeInterval": {"@id": "event:hasTimeInterval", "@type": "TimeInterval"},
            "dateTime": {"@id": "time:inXSDDateTime", "@type": "xsd:dateTime"},
            
            "participants": {"@id": "event:hasParticipant", "@container": "@set"},
            "role": {"@id": "event:hasRole", "@type": "xsd:string"},
            
            "before": {"@id": "temp:before", "@type": "@id"},
            "after": {"@id": "temp:after", "@type": "@id"},
            "contains": {"@id": "temp:contains", "@type": "@id"},
            "during": {"@id": "temp:during", "@type": "@id"}
        }
    }

In [ ]:
#| export
def event_to_jsonld(event, include_temporal_relations=True):
    "Convert an Event to JSON-LD, optionally including inferred temporal relationships"
    # Convert to dict using pydantic
    event_dict = json.loads(event.model_dump_json())
    
    # Add JSON-LD identifiers
    event_dict["@id"] = event_dict.pop("id")
    event_dict["@type"] = "Event"
    
    # If requested, add temporal relationships by inferring them
    if include_temporal_relations:
        # Convert to graph and apply inference
        event_graph = event.to_graph()
        inferred_graph = infer_temporal_relations(event_graph)
        
        # Extract temporal relationships
        temporal_relations = []
        query = """
        PREFIX temp: <https://example.org/temporal-relations#>
        PREFIX event: <https://example.org/event-ontology#>
        
        SELECT ?event1 ?relation ?event2
        WHERE {
            ?event1 event:hasTimeInterval ?interval1 .
            ?interval1 ?relation ?interval2 .
            ?event2 event:hasTimeInterval ?interval2 .
            FILTER(STRSTARTS(STR(?relation), STR(temp:)))
        }
        """
        results = inferred_graph.query(query)
        
        for row in results:
            relation = str(row.relation).split('#')[-1]
            event1 = str(row.event1)
            event2 = str(row.event2)
            
            temporal_relations.append({
                "event1": event1,
                "relation": relation,
                "event2": event2
            })
        
        if temporal_relations:
            event_dict["temporalRelations"] = temporal_relations
    
    # Add context
    event_dict["@context"] = create_jsonld_context()["@context"]
    
    return event_dict

In [ ]:
#| export
def create_test_events():
    "Create test events for demonstrating temporal reasoning"
    # Morning Meeting
    morning_start = TimeInstant(dateTime=datetime(2025, 6, 15, 9, 0, tzinfo=timezone.utc))
    morning_end = TimeInstant(dateTime=datetime(2025, 6, 15, 10, 0, tzinfo=timezone.utc))
    morning_meeting = Event(
        id="event:meeting1",
        name="Morning Meeting",
        description="Team standup meeting",
        startTime=morning_start,
        endTime=morning_end,
        location="event:room101"
    )
    morning_meeting.add_participant("person:alice", "Alice Rodriguez", "Facilitator")
    
    # Afternoon Workshop
    afternoon_start = TimeInstant(dateTime=datetime(2025, 6, 15, 14, 0, tzinfo=timezone.utc))
    afternoon_end = TimeInstant(dateTime=datetime(2025, 6, 15, 16, 0, tzinfo=timezone.utc))
    afternoon_workshop = Event(
        id="event:workshop1",
        name="Afternoon Workshop",
        description="Technical design workshop",
        startTime=afternoon_start,
        endTime=afternoon_end,
        location="event:room202"
    )
    afternoon_workshop.add_participant("person:bob", "Bob Smith", "Presenter")
    
    # Add an instant reification to the workshop
    decision_time = TimeInstant(dateTime=datetime(2025, 6, 15, 15, 30, tzinfo=timezone.utc))
    afternoon_workshop.add_instant_reification(
        "person:bob", "https://example.org/event-ontology#makes", "decision:architecture",
        decision_time,
        {"decision": "Use microservices", "impact": "high"}
    )
    
    return [morning_meeting, afternoon_workshop]

In [ ]:
# Test case 1: Basic time interval representation
now = datetime.now(timezone.utc)
later = now + timedelta(hours=2)

time_instant = TimeInstant(dateTime=now, description="Test time point")
time_interval = TimeInterval(
    startTime=TimeInstant(dateTime=now),
    endTime=TimeInstant(dateTime=later),
    description="Test interval"
)

# Test TimeInstant
test_eq(time_instant.dateTime, now)
test_eq(time_instant.description, "Test time point")

# Test TimeInterval
test_eq(time_interval.startTime.dateTime, now)
test_eq(time_interval.endTime.dateTime, later)
test_eq(time_interval.duration().total_seconds(), 7200)  # 2 hours in seconds

In [ ]:
# Test case 3: Conversion to RDF graph
# Create an event for testing
now = datetime.now(timezone.utc)
later = now + timedelta(hours=2)

test_event = Event(
    id="event:test1",
    name="Test Event",
    description="Event for testing",
    startTime=TimeInstant(dateTime=now),
    endTime=TimeInstant(dateTime=later),
    location="location:test"
)

event_graph = test_event.to_graph()

# Test graph structure
test_eq(isinstance(event_graph, Graph), True)

# Count number of triples
triple_count = len(event_graph)
test_ne(triple_count, 0)  # Graph should not be empty

# Verify event type triple
event_uri = URIRef("event:test1")
event_type_triple = (event_uri, RDF.type, Namespaces.EVENT.Event)
test_eq(event_type_triple in event_graph, True)

In [ ]:
# Test case 4: Temporal inference
# Create test events
test_events = create_test_events()

# Combine graphs
combined_graph = Graph()
for event in test_events:
    combined_graph += event.to_graph()

# Apply temporal inference
inferred_graph = infer_temporal_relations(combined_graph)

# Check for temporal relationships
morning_uri = URIRef("event:meeting1")
afternoon_uri = URIRef("event:workshop1")

# Search for inferred relationships
temporal_relationships = []
query = """
PREFIX temp: <https://example.org/temporal-relations#>
PREFIX event: <https://example.org/event-ontology#>

SELECT ?event1 ?relation ?event2
WHERE {
    ?event1 event:hasTimeInterval ?interval1 .
    ?interval1 ?relation ?interval2 .
    ?event2 event:hasTimeInterval ?interval2 .
    FILTER(STRSTARTS(STR(?relation), STR(temp:)))
}
"""
results = inferred_graph.query(query)

for row in results:
    relation = str(row.relation).split('#')[-1]
    temporal_relationships.append((str(row.event1), relation, str(row.event2)))

# We expect morning meeting to be before afternoon workshop
expected_relationship = ("event:meeting1", "before", "event:workshop1")
test_eq(expected_relationship in temporal_relationships, True)

In [ ]:
# Test case 5: JSON-LD conversion
# Create test events and convert to JSON-LD
test_events = create_test_events()
event_jsonld = event_to_jsonld(test_events[0])

# Test JSON-LD structure
test_eq(isinstance(event_jsonld, dict), True)
test_eq("@context" in event_jsonld, True)
test_eq(event_jsonld["@type"], "Event")
test_eq(event_jsonld["@id"], "event:meeting1")

# Print a sample of the JSON-LD
print("JSON-LD sample for event:")
print(json.dumps({k: event_jsonld[k] for k in ["@id", "@type", "name", "startTime"]}, indent=2))

In [ ]:
# Test case 5: JSON-LD conversion
# Create test events and convert to JSON-LD
test_events = create_test_events()
event_jsonld = event_to_jsonld(test_events[0])

# Test JSON-LD structure
test_eq(isinstance(event_jsonld, dict), True)
test_eq("@context" in event_jsonld, True)
test_eq(event_jsonld["@type"], "Event")
test_eq(event_jsonld["@id"], "event:meeting1")

# Print a sample of the JSON-LD
print("JSON-LD sample for event:")
print(json.dumps({k: event_jsonld[k] for k in ["@id", "@type", "name", "startTime"]}, indent=2))

In [ ]:
# Test case 6: Convert multiple events with temporal relationships
test_events = create_test_events()
multi_event_jsonld = [event_to_jsonld(event) for event in test_events]

# Check if temporal relationships are included
has_temporal_relations = any("temporalRelations" in event for event in multi_event_jsonld)
print(f"Events contain temporal relations: {has_temporal_relations}")

# Print the temporal relationships if present
for event in multi_event_jsonld:
    if "temporalRelations" in event:
        print(f"Temporal relations for {event['@id']}:")
        for relation in event["temporalRelations"]:
            print(f"  {relation['event1']} {relation['relation']} {relation['event2']}")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()